# 12 — Service layer (`factory_floor/services.py`)

Phase 2 of the professionalization work extracted the business logic of a diagnostic turn out of `app.py` into `factory_floor/services.py` — plain functions, no Streamlit. This notebook runs a turn through that layer end to end (blocking **and** streaming) against the real vector store, with `assert` checks, so the `nbconvert` sweep catches any regression in the extraction.

It mirrors notebook 07, but calls `services.run_diagnostic()` instead of `agent.run_diagnostic_agent()` directly.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from factory_floor import services
from factory_floor.config import COLLECTION_NAME, VECTOR_DIR
from factory_floor.vectorstore import get_embeddings, load_vectorstore

vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=get_embeddings())
print('vector store loaded:', vectorstore._collection.count(), 'chunks')

## Blocking turn

A real VFD fault-code question, GENERAL machine (no history tool). The agent should call `search_manuals` and cite a real page.

In [ ]:
req = services.DiagnosticRequest(
    question_text='F30021 ground fault after several hours of running — what should be checked?',
    machine_id='GENERAL',
    equipment_type='VFD',
    language='English',
)
result = services.run_diagnostic(req, vectorstore=vectorstore)

print(result.answer[:800])
print('\n--- tools used ---')
for entry in (result.tool_trace or []):
    print(entry['tool'], entry.get('input'))
print('\nrun_id:', result.run_id)

In [ ]:
assert isinstance(result, services.DiagnosticResult)
assert result.answer and len(result.answer) > 50
assert result.run_id
assert result.blocked is False and result.cache_hit is False
assert any(e['tool'] == 'search_manuals' for e in (result.tool_trace or [])), 'expected a manual search for a fault code'
assert result.documents, 'expected retrieved documents'
print('blocking turn OK')

## Streaming turn

Same request, `stream=True`. The result object stays empty until the generator is fully consumed (exactly how `app.py` drives it via `st.write_stream`).

In [ ]:
generator, streamed_result = services.run_diagnostic(req, vectorstore=vectorstore, stream=True)
assert streamed_result.answer is None, 'result must be empty before the generator is consumed'

chunks = list(generator)
streamed_text = ''.join(chunks)
print(streamed_text[:800])

In [ ]:
assert streamed_result.answer, 'answer should be populated after consuming the generator'
assert streamed_result.answer.strip() == streamed_text.strip() or streamed_text in streamed_result.answer
assert streamed_result.run_id
print('streaming turn OK')

## Turn assembly

The dict `app.py` appends to its conversation list.

In [ ]:
turn = services.assemble_turn(
    streamed_result, image_bytes=None, classification=None, vision_context=None, language='English'
)
assert set(turn) == {
    'type', 'question', 'answer', 'documents', 'sources', 'tool_trace',
    'image_bytes', 'vision_context', 'predicted_label', 'is_defective', 'language',
}
assert turn['type'] == 'agent'
assert turn['answer'] == streamed_result.answer
turn['question'], turn['language'], len(turn['answer'])

## Typo pre-check

`services.check_typo` is the pre-API guard `app.py` runs before submitting.

In [ ]:
assert services.check_typo('display reads F3OO21') == [('F3OO21', 'F30021')]
assert services.check_typo('the motor is overheating') == []
print('typo pre-check OK')